In [1]:
"""
Agentic Search - POC Notebook Pipeline
=======================================
This script serves as the proof-of-concept, structured like notebook cells.
Each "cell" is a self-contained step that can be run independently.
 
To run: Set ANTHROPIC_API_KEY and TAVILY_API_KEY env vars, then:
    python notebooks/poc_pipeline.py
 
Or import cells individually for interactive exploration.
"""
 
import sys
import os
import json

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, project_root)

In [2]:
def cell_1_setup():
    """Initialize configuration and validate API keys."""
    from src.config import Config
 
    config = Config()
 
    print("=== Configuration ===")
    print(f"LLM Model:          {config.llm_model}")
    print(f"Max Search Results:  {config.max_search_results}")
    print(f"Max Entities:        {config.max_entities}")
    print(f"Max Content Chars:   {config.max_content_chars}")
    print(f"Anthropic Key:       {'✓ set' if config.anthropic_api_key else '✗ MISSING'}")
    print(f"Tavily Key:          {'✓ set' if config.tavily_api_key else '✗ MISSING'}")
 
    config.validate()
    print("\n✓ All API keys present. Ready to go!")
    return config

In [3]:
# Cell 1: Setup
config = cell_1_setup()

=== Configuration ===
LLM Model:          claude-haiku-4-5-20251001
Max Search Results:  10
Max Entities:        50
Max Content Chars:   15000
Anthropic Key:       ✓ set
Tavily Key:          ✓ set

✓ All API keys present. Ready to go!


In [4]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELL 2: Load Query Dataset                                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def cell_2_load_queries():
    """Load and explore the query dataset."""
    # data_path = os.path.join(
    #     os.path.dirname(os.path.dirname(os.path.abspath(__file__))),
    #     "data", "queries.json"
    # )
    data_path = os.path.join(
        os.path.abspath(os.path.join(os.getcwd(), "..")),
        "data", "queries.json"
    )
    with open(data_path) as f:
        dataset = json.load(f)
 
    train = dataset["train"]
    eval_set = dataset["eval"]
 
    print(f"=== Query Dataset ===")
    print(f"Train queries:    {len(train)}")
    print(f"Eval queries:     {len(eval_set)}")
 
    # Complexity breakdown
    complexities = {}
    for q in train:
        c = q.get("complexity", "unknown")
        complexities[c] = complexities.get(c, 0) + 1
    print(f"\nTrain complexity: {complexities}")
 
    # Category breakdown
    categories = {}
    for q in train:
        cat = q.get("category", "unknown")
        categories[cat] = categories.get(cat, 0) + 1
    print(f"Train categories: {json.dumps(categories, indent=2)}")
 
    # Show a few examples
    print("\n--- Sample queries ---")
    for q in train[:5]:
        print(f"  [{q['complexity']:8s}] {q['query']}")
        if q.get('post_filters'):
            print(f"             → search: '{q['search_terms']}', filter: {q['post_filters']}")
 
    return dataset

In [5]:
# Cell 2: Load queries
dataset = cell_2_load_queries()

=== Query Dataset ===
Train queries:    40
Eval queries:     15

Train complexity: {'simple': 27, 'complex': 7, 'moderate': 6}
Train categories: {
  "tech_startups": 8,
  "restaurants_food": 6,
  "tools_software": 6,
  "services": 4,
  "education": 4,
  "travel": 3,
  "finance": 2,
  "entertainment": 2,
  "healthcare": 2,
  "shopping": 2,
  "sports": 1
}

--- Sample queries ---
  [simple  ] AI startups in healthcare
  [simple  ] top pizza places in Brooklyn
  [simple  ] open source database tools
  [simple  ] best coworking spaces in Austin
  [simple  ] top machine learning conferences 2025


In [6]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELL 3: Test Query Classification (Stage A)                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def cell_3_test_classification(config):
    """Test the query classifier on sample queries."""
    from src.query_classifier import classify_query
    from src.config import Timer
 
    test_queries = [
        # Should be accepted (topic queries)
        "AI startups in healthcare",
        "best pizza places in Brooklyn",
        "Asian restaurants in Chicago that serve halal food",
        # Should be rejected (not topic queries)
        "What is photosynthesis?",
        "How do I fix a leaky faucet?",
    ]
 
    print("=== Query Classification Tests ===\n")
    for query in test_queries:
        with Timer(f"Classify: {query}"):
            result = classify_query(query, config)
 
        status = "✓ TOPIC" if result.is_topic_query else "✗ REJECT"
        print(f"  [{status}] {query}")
        if result.is_topic_query:
            print(f"    Entity type: {result.entity_type}")
            print(f"    Search:      {result.search_terms}")
            if result.post_filters:
                print(f"    Filters:     {result.post_filters}")
            print(f"    Columns:     {result.suggested_columns}")
        print(f"    Reasoning:   {result.reasoning}")
        print()
 
    return True

In [7]:
# Cell 3: Classification
cell_3_test_classification(config)

=== Query Classification Tests ===



13:22:37 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
13:22:37 [INFO] agentic_search: [Timer] Classify: AI startups in healthcare: 1.87s


  [✓ TOPIC] AI startups in healthcare
    Entity type: company
    Search:      AI startups in healthcare
    Columns:     ['Company Name', 'Founded Year', 'Headquarters', 'Focus Area', 'Funding Raised', 'Key Products/Services', 'Stage']
    Reasoning:   This is a clear topic query seeking a list of companies. The user wants to see multiple AI startups operating in the healthcare sector. Both the primary category (startups) and vertical (healthcare) are straightforward search terms. No complex post-filters are needed as the core constraints are searchable directly.



13:22:39 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
13:22:39 [INFO] agentic_search: [Timer] Classify: best pizza places in Brooklyn: 1.72s


  [✓ TOPIC] best pizza places in Brooklyn
    Entity type: restaurant
    Search:      best pizza places in Brooklyn
    Columns:     ['name', 'neighborhood', 'style', 'rating', 'address', 'phone', 'website']
    Reasoning:   This is a clear topic query seeking a list of pizza restaurants in a specific location. The query is straightforward with no complex filters that require post-processing—'best' is a common search ranking factor, and 'pizza' and 'Brooklyn' are primary categories that search engines handle well.



...

13:22:42 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
13:22:42 [INFO] agentic_search: [Timer] Classify: What is photosynthesis?: 1.30s


  [✗ REJECT] What is photosynthesis?
    Reasoning:   This is a factual/definitional question asking for an explanation of a biological process, not a request for a list of entities. It seeks a single comprehensive answer rather than a structured table of items.



13:22:43 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
13:22:43 [INFO] agentic_search: [Timer] Classify: How do I fix a leaky faucet?: 1.26s


  [✗ REJECT] How do I fix a leaky faucet?
    Reasoning:   This is a how-to/instructional question seeking procedural guidance, not a list of entities. The user wants steps to repair something, not a table of options to choose from.



True

In [8]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELL 4: Test Web Search (Stage B)                                      ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def cell_4_test_search(config):
    """Test Tavily web search on a sample query."""
    from src.web_searcher import search_web
    from src.config import Timer
 
    query = "AI startups in healthcare"
 
    print(f"=== Web Search Test: '{query}' ===\n")
    with Timer("Web Search"):
        results = search_web(query, config)
 
    print(f"Found {len(results)} results:\n")
    for i, r in enumerate(results):
        print(f"  {i+1}. {r.title}")
        print(f"     URL: {r.url}")
        print(f"     Score: {r.score:.3f}")
        print(f"     Content length: {len(r.content)} chars")
        print(f"     Snippet: {r.snippet[:120]}...")
        print()
 
    return results

In [9]:
# Cell 4: Search
results = cell_4_test_search(config)

=== Web Search Test: 'AI startups in healthcare' ===



13:22:47 [INFO] agentic_search: Search returned 10 results for: AI startups in healthcare
13:22:47 [INFO] agentic_search: [Timer] Web Search: 3.80s


Found 10 results:

  1. Breakthrough AI Startups Making Waves in Healthcare in 2026
     URL: https://www.solutelabs.com/blog/top-ai-healthcare-startups
     Score: 0.929
     Content length: 2343 chars
     Snippet: These AI healthcare startups represent the diversity of applications that artificial intelligence applies in healthcare—...

  2. Roadmap: Healthcare AI - Bessemer Venture Partners
     URL: https://www.bvp.com/atlas/roadmap-healthcare-ai
     Score: 0.887
     Content length: 2275 chars
     Snippet: Healthcare AI has become a focal point for US policymakers and legislators, prompting a fundamental reassessment of long...

  3. Top Artificial Intelligence Companies in Healthcare
     URL: https://medicalfuturist.com/top-artificial-intelligence-companies-in-healthcare
     Score: 0.823
     Content length: 1892 chars
     Snippet: ## Health Management

### 1) Ada Health

Ada is a health company based in Berlin that operates an end-user self-assessme...

  4. Healthcare IT 

In [10]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELL 5: Test Content Enrichment (Stage C)                              ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def cell_5_test_enrichment(results, config):
    """Test content enrichment/scraping on search results."""
    from src.content_parser import enrich_content
    from src.config import Timer
 
    print("=== Content Enrichment Test ===\n")
 
    # Show pre-enrichment stats
    for i, r in enumerate(results[:3]):
        print(f"  Before enrichment #{i+1}: {len(r.content)} chars")
 
    with Timer("Content Enrichment"):
        enriched = enrich_content(results, config)
 
    print()
    for i, r in enumerate(enriched[:3]):
        print(f"  After enrichment #{i+1}: {len(r.content)} chars")
        print(f"    Preview: {r.content[:150]}...")
        print()
 
    return enriched

In [11]:
# Cell 5: Enrichment
enriched = cell_5_test_enrichment(results, config)

13:22:47 [INFO] agentic_search: Content enrichment: 10 results, 0 additionally scraped
13:22:47 [INFO] agentic_search: [Timer] Content Enrichment: 0.00s


=== Content Enrichment Test ===

  Before enrichment #1: 2343 chars
  Before enrichment #2: 2275 chars
  Before enrichment #3: 1892 chars

  After enrichment #1: 2343 chars
    Preview: These AI healthcare startups represent the diversity of applications that artificial intelligence applies in healthcare—from medical imaging and facil...

  After enrichment #2: 2275 chars
    Preview: Healthcare AI has become a focal point for US policymakers and legislators, prompting a fundamental reassessment of long-standing healthcare regulatio...

  After enrichment #3: 1892 chars
    Preview: ## Health Management

### 1) Ada Health

Ada is a health company based in Berlin that operates an end-user self-assessment app. The app started out as...



In [12]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELL 6: Test Entity Extraction (Stage C2)                              ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def cell_6_test_extraction(enriched_results, config):
    """Test LLM-based entity extraction."""
    from src.entity_extractor import extract_entities
    from src.config import QueryAnalysis, Timer
    import pandas as pd
 
    query = "AI startups in healthcare"
    analysis = QueryAnalysis(
        is_topic_query=True,
        entity_type="company",
        search_terms="AI startups in healthcare",
        post_filters=[],
        # suggested_columns=["name", "description", "focus_area", "funding", "location"],
        # suggested_columns=['name', 'description', 'location', 'funding_stage', 'founding_year', 'focus_area', 'website']
        suggested_columns=['Company Name', 'Founding Year', 'Focus Area', 'Funding Status', 'Location', 'Key Technology']
    )
 
    print(f"=== Entity Extraction Test: '{query}' ===\n")
    with Timer("Entity Extraction"):
        entities, columns = extract_entities(query, analysis, enriched_results, config)
 
    print(f"Extracted {len(entities)} entities with columns: {columns}\n")
 
    if entities:
        rows = [e.attributes for e in entities]
        df = pd.DataFrame(rows)
        ordered = [c for c in columns if c in df.columns]
        df = df[ordered]
        print(df.to_string(index=False))
 
        print("\n--- Source tracing (first 3 entities) ---")
        for e in entities[:3]:
            name = e.attributes.get("name", "?")
            sources = set(v for v in e.sources.values() if v)
            print(f"  {name}: {sources}")
 
    return entities, columns

In [13]:
# Cell 6: Extraction
entities, columns = cell_6_test_extraction(enriched, config)

=== Entity Extraction Test: 'AI startups in healthcare' ===



13:23:08 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
13:23:08 [INFO] agentic_search: Extracted 32 entities with 6 columns
13:23:08 [INFO] agentic_search: [Timer] Entity Extraction: 21.44s


Extracted 32 entities with columns: ['name', 'Founding Year', 'Focus Area', 'Funding Status', 'Location', 'Key Technology']

                     name  Founding Year                                                 Focus Area                            Funding Status                        Location                                                              Key Technology
               Biofourmis            NaN    Remote patient monitoring, chronic condition management                                       NaN                             NaN     FDA-cleared algorithms, clinical-grade wearables, virtual care platform
               Ada Health            NaN                      Health management, symptom assessment                                       NaN                          Berlin      Self-assessment app, symptom matching, statistical likelihood analysis
                    Nabla            NaN                  Medical documentation, clinical workflows                          

In [14]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELL 7: Full Pipeline Test                                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def cell_7_full_pipeline(config):
    """Run the complete pipeline on several test queries."""
    from src.pipeline import run, display_result
 
    test_queries = [
        "open source database tools",
        "best coffee roasters in Portland",
        # Complex query with post-filter:
        # "vegan restaurants in Los Angeles with outdoor seating",
    ]
 
    print("=" * 70)
    print("FULL PIPELINE TESTS")
    print("=" * 70)
 
    for query in test_queries:
        print(f"\n{'─' * 70}")
        result = run(query, config)
        print(display_result(result))
        print(f"{'─' * 70}\n")
 
    return True

In [15]:
# Cell 7: Full pipeline
cell_7_full_pipeline(config)

FULL PIPELINE TESTS

──────────────────────────────────────────────────────────────────────


13:23:10 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
13:23:10 [INFO] agentic_search: [Timer] Stage A: Query Classification: 1.61s
13:23:10 [INFO] agentic_search: Query accepted: entity_type=tool, search='open source database tools', post_filters=[]
13:23:10 [INFO] agentic_search: Search returned 10 results for: open source database tools
13:23:10 [INFO] agentic_search: [Timer] Stage B: Web Search: 0.16s
13:23:10 [INFO] agentic_search: Content enrichment: 10 results, 0 additionally scraped
13:23:10 [INFO] agentic_search: [Timer] Stage C: Content Enrichment: 0.00s
13:23:30 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
13:23:30 [INFO] agentic_search: Extracted 20 entities with 6 columns
13:23:30 [INFO] agentic_search: [Timer] Stage C2: Entity Extraction: 19.29s
13:23:30 [INFO] agentic_search: Pipeline complete: 20 entities, 6 columns, 21.06s


Query: open source database tools
Is topic query: True
Entity type: tool
Search terms: open source database tools
Search results: 10
Entities found: 20
Processing time: 21.06s

                              name                                                                                                                                                                                                                                                                                                               description primary_language            license                                                                                                                       use_cases github_stars
             DB Browser for SQLite                                  A visual interface tool for creating, designing, and editing SQLite database files without requiring SQL knowledge. Includes tools for creating and modifying tables and indexes, editing data in a spreadsheet-like interface, impor

13:23:32 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
13:23:32 [INFO] agentic_search: [Timer] Stage A: Query Classification: 2.76s
13:23:32 [INFO] agentic_search: Query accepted: entity_type=company, search='best coffee roasters in Portland', post_filters=[]
13:23:32 [INFO] agentic_search: Search returned 10 results for: best coffee roasters in Portland
13:23:32 [INFO] agentic_search: [Timer] Stage B: Web Search: 0.17s
13:23:32 [INFO] agentic_search: Content enrichment: 10 results, 0 additionally scraped
13:23:32 [INFO] agentic_search: [Timer] Stage C: Content Enrichment: 0.00s
13:23:47 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
13:23:47 [INFO] agentic_search: Extracted 24 entities with 6 columns
13:23:47 [INFO] agentic_search: [Timer] Stage C2: Entity Extraction: 14.48s
13:23:47 [INFO] agentic_search: Pipeline complete: 24 entities, 6 columns, 17.41s


Query: best coffee roasters in Portland
Is topic query: True
Entity type: company
Search terms: best coffee roasters in Portland
Search results: 10
Entities found: 24
Processing time: 17.41s

                     name                                            location                                                                specialty  founded                                  website rating
         Never Coffee Lab                      Hawthorne & Downtown, Portland                   Fruit-forward blends with some single-origin offerings      NaN               https://nevercoffeelab.com   None
 Sterling Coffee Roasters                               NW District, Portland       Single origin from Latin America and Ethiopia, well-rounded blends      NaN             https://www.sterling.coffee/   None
          Marigold Coffee                                            Portland                                Single origin beans from around the world      NaN                         

True

In [16]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELL 8: Post-Filter Test (Complex Query)                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def cell_8_test_post_filter(config):
    """Test the full pipeline on a complex query with post-filtering."""
    from src.pipeline import run, display_result
 
    query = "Asian restaurants in Chicago that serve halal food"
 
    print(f"=== Complex Query Test: '{query}' ===\n")
    result = run(query, config)
    print(display_result(result))
 
    return result

In [17]:
# Cell 8: Complex query
# Uncomment to test (uses extra API credits):
cell_8_test_post_filter(config)

=== Complex Query Test: 'Asian restaurants in Chicago that serve halal food' ===



13:23:49 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
13:23:49 [INFO] agentic_search: [Timer] Stage A: Query Classification: 1.92s
13:23:49 [INFO] agentic_search: Query accepted: entity_type=restaurant, search='Asian restaurants in Chicago', post_filters=['serves halal food']
13:23:49 [INFO] agentic_search: Search returned 10 results for: Asian restaurants in Chicago
13:23:49 [INFO] agentic_search: [Timer] Stage B: Web Search: 0.18s
13:23:49 [INFO] agentic_search: Content enrichment: 10 results, 0 additionally scraped
13:23:49 [INFO] agentic_search: [Timer] Stage C: Content Enrichment: 0.00s
13:24:02 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
13:24:02 [INFO] agentic_search: Extracted 23 entities with 7 columns
13:24:02 [INFO] agentic_search: [Timer] Stage C2: Entity Extraction: 13.37s
13:24:02 [INFO] agentic_search: Applying post-filters: ['serves halal food'] to 23 entities
13:24:07 [INFO] httpx:

Query: Asian restaurants in Chicago that serve halal food
Is topic query: True
Entity type: restaurant
Search terms: Asian restaurants in Chicago
Post-filters: ['serves halal food']
Search results: 10
Entities found: 0
Processing time: 20.16s


PipelineResult(query='Asian restaurants in Chicago that serve halal food', analysis=QueryAnalysis(is_topic_query=True, entity_type='restaurant', search_terms='Asian restaurants in Chicago', post_filters=['serves halal food'], suggested_columns=['name', 'cuisine type', 'address', 'phone', 'website', 'halal certification', 'rating'], reasoning="This is a valid topic query seeking a list of restaurants. The core search (Asian restaurants in Chicago) uses primary categories and location. Halal certification is a specific dietary/religious attribute that's uncommon and not reliably filtered by search engines, so it's better retrieved as a post-filter to verify after getting initial results."), entities=[], columns=['name', 'cuisine_type', 'address', 'phone', 'website', 'halal_certification', 'rating'], search_results_count=10, processing_time=20.15540599822998, errors=[])

In [18]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELL 8: Post-Filter Test (Complex Query)                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def cell_8_test_post_filter(config):
    """Test the full pipeline on a complex query with post-filtering."""
    from src.pipeline import run, display_result
 
    query = "Indian veg-only resteraunts in Chicago"
 
    print(f"=== Complex Query Test: '{query}' ===\n")
    result = run(query, config)
    print(display_result(result))
 
    return result

In [19]:
cell_8_test_post_filter(config)

=== Complex Query Test: 'Indian veg-only resteraunts in Chicago' ===



13:24:09 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
13:24:09 [INFO] agentic_search: [Timer] Stage A: Query Classification: 2.21s
13:24:09 [INFO] agentic_search: Query accepted: entity_type=restaurant, search='vegetarian Indian restaurants in Chicago', post_filters=['veg-only (no non-vegetarian options)', 'serves Indian cuisine']
13:24:10 [INFO] agentic_search: Search returned 10 results for: vegetarian Indian restaurants in Chicago
13:24:10 [INFO] agentic_search: [Timer] Stage B: Web Search: 0.26s
13:24:10 [INFO] agentic_search: Scraping: https://www.reddit.com/r/chicagovegan/comments/16gro2w/chicago_vegans_where_do_you_go_for_indian_food/
13:24:10 [WARNING] trafilatura.core: discarding data: None
13:24:10 [WARNING] agentic_search: Trafilatura extraction failed for https://www.reddit.com/r/chicagovegan/comments/16gro2w/chicago_vegans_where_do_you_go_for_indian_food/
13:24:10 [INFO] agentic_search: Content enrichment: 10 results, 0 additiona

Query: Indian veg-only resteraunts in Chicago
Is topic query: True
Entity type: restaurant
Search terms: vegetarian Indian restaurants in Chicago
Post-filters: ['veg-only (no non-vegetarian options)', 'serves Indian cuisine']
Search results: 10
Entities found: 1
Processing time: 11.63s

                       name neighborhood cuisine price_range  rating phone                   website
Annapurna Simply Vegetarian  Rogers Park  Indian          $$     7.9  None https://eatannapurna.com/

--- Sources ---
  Annapurna Simply Vegetarian: https://www.theinfatuation.com/chicago/guides/best-indian-restaurants-chicago


PipelineResult(query='Indian veg-only resteraunts in Chicago', analysis=QueryAnalysis(is_topic_query=True, entity_type='restaurant', search_terms='vegetarian Indian restaurants in Chicago', post_filters=['veg-only (no non-vegetarian options)', 'serves Indian cuisine'], suggested_columns=['name', 'neighborhood', 'cuisine', 'price_range', 'rating', 'phone', 'website'], reasoning="User is seeking a list of restaurants matching specific criteria. 'Indian' and 'vegetarian' are primary cuisine/dietary attributes that should be in the search query. 'Veg-only' (completely vegetarian, no meat options) is a stricter filter than just 'vegetarian' and may need post-filtering to verify establishments don't serve any non-veg dishes."), entities=[Entity(attributes={'name': 'Annapurna Simply Vegetarian', 'neighborhood': 'Rogers Park', 'cuisine': 'Indian', 'price_range': '$$', 'rating': 7.9, 'phone': None, 'website': 'https://eatannapurna.com/'}, sources={'name': 'https://www.theinfatuation.com/chicago

In [20]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELL 9: Review Bomb Test                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
 
def cell_9_test_review_bomb(config):
    """
    Test the review bomb feature end-to-end.
    
    Runs a query that produces entities with clear popularity differences,
    then re-ranks them using review/popularity signals.
    """
    from src.pipeline import run, display_result
    from src.config import Timer
    import pandas as pd
 
    query = "best pizza places in Brooklyn"
 
    print("=" * 70)
    print("REVIEW BOMB TEST")
    print("=" * 70)
 
    # First: run WITHOUT review bomb for baseline
    print(f"\n--- Baseline (no review bomb): '{query}' ---\n")
    with Timer("Pipeline without review bomb"):
        baseline = run(query, config, enable_review_bomb=False)
 
    if baseline.entities:
        rows = [e.attributes for e in baseline.entities]
        df = pd.DataFrame(rows)
        if "name" in df.columns:
            print("Baseline order:")
            for i, name in enumerate(df["name"]):
                print(f"  {i+1}. {name}")
    else:
        print("No entities found — skipping review bomb test.")
        return None
 
    # Then: run WITH review bomb
    print(f"\n--- With Review Bomb: '{query}' ---\n")
    with Timer("Pipeline with review bomb"):
        bombed = run(query, config, enable_review_bomb=True)
 
    print(display_result(bombed))
 
    # Show the re-ranking diff
    if bombed.entities:
        print("\n--- Re-ranking Summary ---")
        for i, entity in enumerate(bombed.entities):
            name = entity.attributes.get("name", "?")
            score = entity.attributes.get("popularity_score", "?")
            summary = entity.attributes.get("review_summary", "")
            print(f"  {i+1}. [{score}/100] {name}")
            print(f"     {summary}")
 
    return bombed

In [21]:
# Cell 9: Review bomb
cell_9_test_review_bomb(config)

REVIEW BOMB TEST

--- Baseline (no review bomb): 'best pizza places in Brooklyn' ---



13:24:20 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
13:24:20 [INFO] agentic_search: [Timer] Stage A: Query Classification: 1.54s
13:24:20 [INFO] agentic_search: Query accepted: entity_type=restaurant, search='best pizza places in Brooklyn', post_filters=[]
13:24:24 [INFO] agentic_search: Search returned 10 results for: best pizza places in Brooklyn
13:24:24 [INFO] agentic_search: [Timer] Stage B: Web Search: 3.87s
13:24:24 [INFO] agentic_search: Scraping: https://www.tripadvisor.com/Restaurants-g60827-c31-Brooklyn_New_York.html
13:24:24 [WARNING] agentic_search: Scrape failed for https://www.tripadvisor.com/Restaurants-g60827-c31-Brooklyn_New_York.html: 403 Client Error: Forbidden for url: https://www.tripadvisor.com/Restaurants-g60827-c31-Brooklyn_New_York.html
13:24:24 [INFO] agentic_search: Scraping: https://www.reddit.com/r/Brooklyn/comments/173q8cs/best_pizza_in_brooklyn/
13:24:24 [WARNING] trafilatura.core: discarding data: None
13:24

Baseline order:
  1. Di Fara
  2. L'Industrie
  3. Lucali
  4. Grimaldi's Pizzeria
  5. Totonno's
  6. L&B Spumoni Gardens
  7. La Villa Pizzeria
  8. Juliana's
  9. Sottocasa Pizzeria
  10. Roberta's
  11. F&F Restaurant
  12. Wheated
  13. Ops
  14. The Astarita
  15. Luigi's Pizza
  16. Espresso Pizzeria
  17. Sabrina's
  18. Paulie Gee's Slice Shop
  19. Fini Pizza
  20. Pizza D'amore
  21. Dellarocco's Brick Oven Pizza
  22. Bklyn Pizza
  23. My Little Pizzeria
  24. Front Street Pizza
  25. Table 87
  26. GANNI'S PIZZERIA

--- With Review Bomb: 'best pizza places in Brooklyn' ---



13:24:42 [INFO] httpx: HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
13:24:42 [INFO] agentic_search: [Timer] Stage A: Query Classification: 2.08s
13:24:42 [INFO] agentic_search: Query accepted: entity_type=restaurant, search='best pizza places in Brooklyn', post_filters=[]
13:24:42 [INFO] agentic_search: Search returned 10 results for: best pizza places in Brooklyn
13:24:42 [INFO] agentic_search: [Timer] Stage B: Web Search: 0.17s
13:24:42 [INFO] agentic_search: Scraping: https://www.tripadvisor.com/Restaurants-g60827-c31-Brooklyn_New_York.html
13:24:42 [WARNING] agentic_search: Scrape failed for https://www.tripadvisor.com/Restaurants-g60827-c31-Brooklyn_New_York.html: 403 Client Error: Forbidden for url: https://www.tripadvisor.com/Restaurants-g60827-c31-Brooklyn_New_York.html
13:24:42 [INFO] agentic_search: Scraping: https://www.reddit.com/r/Brooklyn/comments/173q8cs/best_pizza_in_brooklyn/
13:24:42 [WARNING] trafilatura.core: discarding data: None
13:24

Query: best pizza places in Brooklyn
Is topic query: True
Entity type: restaurant
Search terms: best pizza places in Brooklyn
Search results: 10
Entities found: 26
Processing time: 55.71s

                         name  popularity_score                                                                                                                                                                                review_summary                                                                                                                                                           key_signals    neighborhood    cuisine_style  rating price_range                            address                                                             website
              Juliana's Pizza                87 Highly-rated Neapolitan pizza destination with 4.6 stars on Tripadvisor and consistently praised for exceptional quality, fresh ingredients, and excellent service across multiple locations.              

PipelineResult(query='best pizza places in Brooklyn', analysis=QueryAnalysis(is_topic_query=True, entity_type='restaurant', search_terms='best pizza places in Brooklyn', post_filters=[], suggested_columns=['name', 'neighborhood', 'cuisine_style', 'rating', 'price_range', 'address', 'website'], reasoning="This is a clear topic query seeking a list of pizza restaurants in Brooklyn. The location (Brooklyn) and cuisine type (pizza) are primary search constraints that should be in search_terms. No additional post-filters needed as 'best' is a general quality descriptor that search results will naturally address."), entities=[Entity(attributes={'name': "Juliana's Pizza", 'neighborhood': None, 'cuisine_style': 'Italian, Pizza', 'rating': 4.6, 'price_range': None, 'address': None, 'website': None, 'popularity_score': 87, 'review_summary': 'Highly-rated Neapolitan pizza destination with 4.6 stars on Tripadvisor and consistently praised for exceptional quality, fresh ingredients, and excellent s